In [1]:
import pandas as pd
import os

# Load data

In [2]:
data_path = "dataset/mimic-iv-3.1/hosp/"
checkpoint_path = "dataset/filteredData/"
note_data_path = "dataset/note/"

checkpoint_file = os.path.join(checkpoint_path, 'ami_cohort_structured_features.csv')

Load pre processed strucutred data to filter

In [3]:
try:
    ami_cohort_df = pd.read_csv(checkpoint_file)
    print(f"Successfully loaded pre-processed cohort from: {checkpoint_file}")
except FileNotFoundError:
    print(f"ERROR: Could not find '{checkpoint_file}'.")
    print("Please make sure the file is in the same folder as your notebook.")
    # Stop execution if data isn't loaded
    raise ValueError("Checkpoint DataFrame not available.")

Successfully loaded pre-processed cohort from: dataset/filteredData/ami_cohort_structured_features.csv


In [4]:
cohort_hadm_ids_set = set(ami_cohort_df['hadm_id'])

Load Notes

In [5]:
discharge_notes_path = os.path.join(note_data_path, 'discharge.csv')
chunk_size = 100_000
filtered_note_chunks = []

In [6]:
try:
    # --- 5. Create the iterator ---
    chunk_iterator = pd.read_csv(discharge_notes_path,
                                 chunksize=chunk_size,
                                 low_memory=False,
                                 # We only need hadm_id (to link) and text
                                 usecols=['hadm_id', 'text'])

    # --- 6. Loop through each chunk ---
    for i, chunk in enumerate(chunk_iterator):
        print(f"  Processing chunk {i+1}...")

        # 7. Apply the filter: Keep only rows for our AMI cohort
        chunk.dropna(subset=['hadm_id'], inplace=True)
        chunk['hadm_id'] = chunk['hadm_id'].astype(int)

        filtered_chunk = chunk[chunk['hadm_id'].isin(cohort_hadm_ids_set)]

        # 8. Add this filtered chunk to our list
        if not filtered_chunk.empty:
            filtered_note_chunks.append(filtered_chunk)
            print(
                f"    Found {len(filtered_chunk)} relevant notes in this chunk.")

    print("\nNote file processing complete.")

    # --- 9. Concatenate all the small, filtered chunks ---
    if filtered_note_chunks:
        ami_notes_df = pd.concat(filtered_note_chunks)
        print(
            f"\nSuccessfully created final ami_notes_df with {len(ami_notes_df)} rows.")

        # --- 10. Save this new DataFrame! ---
        # Save it as a new checkpoint
        notes_csv_path = os.path.join(
            checkpoint_path, 'ami_cohort_discharge_notes.csv')
        ami_notes_df.to_csv(notes_csv_path, index=False)
        print(f"Successfully saved all cohort notes to: {notes_csv_path}")
        print(ami_notes_df.head())
    else:
        print("\nNo discharge notes found for the specified cohort.")

except FileNotFoundError:
    print(f"ERROR: Could not find '{discharge_notes_path}'")
    print("Please ensure 'discharge.csv.gz' is in your 'data/' folder.")
except ValueError as e:
    print(f"ERROR: {e}")
    print("This might be a 'usecols' error. Check if 'discharge.csv.gz' contains 'hadm_id' and 'text'.")
except Exception as e:
    print(f"An error occurred: {e}")

  Processing chunk 1...
    Found 8473 relevant notes in this chunk.
  Processing chunk 2...
    Found 8346 relevant notes in this chunk.
  Processing chunk 3...
    Found 8125 relevant notes in this chunk.
  Processing chunk 4...
    Found 2733 relevant notes in this chunk.

Note file processing complete.

Successfully created final ami_notes_df with 27677 rows.
Successfully saved all cohort notes to: dataset/filteredData/ami_cohort_discharge_notes.csv
     hadm_id                                               text
9   27897940   \nName:  ___               Unit No:   ___\n \...
19  26913865   \nName:  ___          Unit No:   ___\n \nAdmi...
20  24947999   \nName:  ___          Unit No:   ___\n \nAdmi...
21  25242409   \nName:  ___          Unit No:   ___\n \nAdmi...
22  25911675   \nName:  ___          Unit No:   ___\n \nAdmi...
